# **Practical No-02: Text Classification using Embedding Layer and LSTM**

**Course Name: GENERATIVE AI**

**Student Name: RAGAV SHARMA**

**Student ID: 202401110012**

**Dataset: SMS Spam Collection Dataset**

**1. Aim**

To build a binary text classification model using an Embedding layer and Long Short-Term Memory (LSTM) network for classifying SMS messages into **Ham (Not Spam)** and **Spam**, and to evaluate the model using appropriate classification metrics.

**2. Theory**

Text Classification is an important Natural Language Processing (NLP) task used to categorize text into predefined classes. In this practical, the **SMS Spam Collection Dataset** is used to classify messages into Ham and Spam.

An **Embedding layer** converts words into dense numerical vectors that can capture relationships between words. An **LSTM (Long Short-Term Memory)** network processes the sequence of word vectors and learns patterns useful for classification.

For binary classification, a **Sigmoid** activation function is used in the final Dense layer. The model is trained using **binary cross-entropy** loss and evaluated using accuracy, precision, recall, F1-score, a classification report, and a confusion matrix.

**Step 1 — Load the Dataset**

In [ ]:
import pandas as pd
import zipfile
import io
import urllib.request

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

data = urllib.request.urlopen(url).read()

with zipfile.ZipFile(io.BytesIO(data)) as z:
    with z.open("SMSSpamCollection") as f:
        df = pd.read_csv(
            f,
            sep="\t",
            header=None,
            names=["Label", "Message"],
            encoding="utf-8"
        )

print("Dataset loaded successfully!")
print("Shape:", df.shape)

**Step 2 — View the Dataset**

In [ ]:
print(df.head())

In [ ]:
print(df.columns)

**Step 3 — Check the Class Distribution**

In [ ]:
print(df["Label"].value_counts())

**Step 4 — Convert Labels into Binary Values**

For our assignment, we will use:

- **ham → 0**
- **spam → 1**

In [ ]:
df["Sentiment"] = df["Label"].map({"ham": 0, "spam": 1})

print("Label distribution:")
print(df["Sentiment"].value_counts())

**Step 5 — Create a Balanced Dataset**

The SMS Spam Collection contains more ham messages than spam messages. To make the classification task balanced, we will select an equal number of messages from both classes.

In [ ]:
positive = df[df["Sentiment"] == 1]
negative = df[df["Sentiment"] == 0]

sample_size = min(len(positive), len(negative))

positive_sample = positive.sample(n=sample_size, random_state=42)
negative_sample = negative.sample(n=sample_size, random_state=42)

df_balanced = pd.concat(
    [positive_sample, negative_sample],
    axis=0
)

df_balanced = df_balanced.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Balanced dataset shape:", df_balanced.shape)
print("\nBalanced class distribution:")
print(df_balanced["Sentiment"].value_counts())

**Step 6 — Keep Only the Columns We Need**

In [ ]:
df_balanced = df_balanced[["Message", "Sentiment"]]

df_balanced.head()

**Step 7 — Clean the SMS Text**

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_balanced["Clean_Text"] = df_balanced["Message"].apply(clean_text)

df_balanced[["Message", "Clean_Text", "Sentiment"]].head()

**Step 8 — Separate X and y**

In [ ]:
X = df_balanced["Clean_Text"]
y = df_balanced["Sentiment"]

print("Number of messages:", len(X))
print("Number of labels:", len(y))

**Step 9 — Split into Training and Testing Data**

80% → Training

20% → Testing

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

**Step 10 — Tokenization**

Now we convert words into numbers because an LSTM cannot directly process normal text.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

MAX_WORDS = 5000

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

print("Vocabulary size:", len(tokenizer.word_index))

**Step 11 — Convert Text to Sequences**

In [ ]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

print("Example sequence:")
print(X_train_seq[0])

**Step 12 — Padding**

Different SMS messages have different lengths. LSTM needs sequences of the same length, so we use padding.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 80

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("Training shape:", X_train_pad.shape)
print("Testing shape:", X_test_pad.shape)

**Step 13 — Build the Embedding + LSTM Model**

The model uses an Input layer, Embedding layer, LSTM layer, Dropout layer, and a Dense sigmoid output layer.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout

VOCAB_SIZE = 5000
EMBEDDING_DIM = 128
LSTM_UNITS = 64

model = Sequential([
    Input(shape=(MAX_LEN,)),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    LSTM(LSTM_UNITS),

    Dropout(0.5),

    Dense(1, activation="sigmoid")
])

model.summary()

**1. Embedding Layer**

Converts word IDs into dense numerical vectors.

**2. LSTM Layer**

Learns sequential patterns and relationships between words in the SMS messages.

**3. Dropout Layer**

Helps reduce overfitting during training.

**4. Dense Layer**

Produces the final binary classification probability using the Sigmoid activation function.

**Step 14 — Compile the Model**

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully!")

**Step 15 — Train the Model**

We will train the model for **10 epochs**, which is more than the 5 epochs used in the reference practical.

In [ ]:
EPOCHS = 10
BATCH_SIZE = 32

history = model.fit(
    X_train_pad,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.20,
    verbose=1
)

**Step 16 — Evaluate on Test Data**

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test,
    verbose=1
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

**Step 17 — Plot Training and Validation Accuracy**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")

plt.legend()
plt.show()

**Step 18 — Plot Training and Validation Loss**

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")

plt.legend()
plt.show()

**Step 19 — Generate Predictions**

In [ ]:
import numpy as np

y_pred_prob = model.predict(X_test_pad)

y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

print("Predictions generated successfully!")
print("Number of predictions:", len(y_pred))

**Step 20 — Classification Metrics**

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Classification Metrics")
print("----------------------")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")

**Step 21 — Detailed Classification Report**

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=["Ham", "Spam"]
))

**Step 22 — Confusion Matrix**

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
plt.imshow(cm)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.xticks(
    [0, 1],
    ["Ham", "Spam"]
)

plt.yticks(
    [0, 1],
    ["Ham", "Spam"]
)

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.show()

**Step 23 — Display the Confusion Matrix Numerically**

In [ ]:
print("Confusion Matrix:")
print(cm)

**Step 24 — Test the Model With Example SMS Messages**

The following function cleans a message, tokenizes it, pads it to the same length used during training, and predicts whether it is Ham or Spam.

In [ ]:
def predict_spam(message):
    cleaned = clean_text(message)

    sequence = tokenizer.texts_to_sequences([cleaned])

    padded = pad_sequences(
        sequence,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

    probability = model.predict(
        padded,
        verbose=0
    )[0][0]

    if probability >= 0.5:
        label = "Spam"
    else:
        label = "Ham"

    return label, probability

**Step 25 — Test With Sample Messages**

In [ ]:
messages = [
    "Congratulations! You have won a free prize. Call now to claim your reward.",
    "Can you send me the notes for today's class?",
    "URGENT! You have been selected for a cash prize. Reply now.",
    "I will reach home by 7 pm. See you soon."
]

for message in messages:
    label, probability = predict_spam(message)

    print("Message:", message)
    print("Prediction:", label)
    print("Spam Probability:", round(float(probability), 4))
    print("-" * 70)

**Step 26 — Take SMS Input From the User**

This cell allows the user to enter any SMS message manually and get a Ham or Spam prediction from the trained LSTM model.

In [ ]:
user_message = input("Enter an SMS message: ")

label, probability = predict_spam(user_message)

print("\nMessage:", user_message)
print("Predicted Class:", label)
print("Spam Probability:", round(float(probability), 4))

# **Final Result and Conclusion**

The **SMS Spam Collection Dataset** was used as a different real-world text classification dataset. SMS messages were converted into binary labels, cleaned, tokenized, converted into padded sequences, and classified using an **Embedding layer and LSTM network**.

The model was trained for **10 epochs** and its performance was evaluated using **accuracy, precision, recall, F1-score, classification report, and a confusion matrix**.

The trained model can also accept a new SMS message from the user and predict whether it is **Ham or Spam**.